In [ ]:
from init import model, embed_model, system_prompt
from langchain_community.document_loaders.markdown import UnstructuredMarkdownLoader
import chromadb

model = model
embed_model = embed_model
system_prompt = system_prompt

#  创建 Chroma 客户端
chroma_client = chromadb.PersistentClient(path="./chroma_db")

collection = chroma_client.get_or_create_collection(
    name="FAQ_collection",
)

from langchain_text_splitters import MarkdownHeaderTextSplitter

with open("../recourses/FAQ/在线学习平台FAQ知识库（智能客服RAG专用）.md", "r", encoding="utf-8") as f:
    markdown_text = f.read()

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),  # 这就是我们需要的二级标题
    # ("###", "Header 3"), # 如果有需要，可以继续添加
]

splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
docs = splitter.split_text(markdown_text)
ids = [f"C{i}" for i in range(len(docs))]


In [ ]:
from rich import print as rprint

rprint(docs)

In [3]:
from langchain_core.messages import HumanMessage, SystemMessage
from init import model, system_prompt
from rich import print as rprint

messages =[
    SystemMessage(system_prompt),
    HumanMessage(""+"\n 如果涉及知识盲区，则在回答末尾处添加“需要检索”")
]

res = model.invoke(messages)
rprint(res)

AIMessage(
    content='哎呀，这个问题可难不倒我，虽然我是学习平台的客服，但闲聊也接得住！明天吃啥，得看你喜欢清淡还是重口，自
己做还是点外卖。要是图省事，可以试试三明治配牛奶；想吃热乎的，来碗番茄鸡蛋面也不错。要是想出门吃，火锅、烤肉、日料
都挺香，就看你的胃和钱包怎么商量啦！',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 
'我们根据规则，用户问“计划中明天吃什么”，这显然不是平台业务问题，是闲聊。可以不用知识库直接回答。但要注意不能使用表
情动作。可以轻松回应。因为不涉及平台具体业务，直接回答即可。但内容要严谨？可以给个合理建议。不需要检索。'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 151,
            'prompt_tokens': 422,
            'total_tokens': 573,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 65,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},
            'prompt_cache_hit_tokens': 384,
            'prompt_cache_miss_tokens': 38
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-v4-flash',
        'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
        'id': '73a71aa5-ace4-4e03-b8c2-157d99f3ebda',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--01a00a19-f665-74b2-b76f-a328ccfc7dc3-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 422,
        'output_tokens': 151,
        'total_tokens': 573,
        'input_token_details': {'cache_read': 384},
        'output_token_details': {'reasoning': 65}
    }
)

In [ ]:
from typing import Annotated
from operator import add


from langgraph.graph.message import MessagesState




class OverAllState(MessagesState):
    input: Annotated[str, add]
    llm_output: Annotated[str,]
    retrieval_output: Annotated[str,]

def llm_node(state: OverAllState) -> OverAllState:
    input_str = state["input"]



In [ ]:
from typing import Annotated
from operator import add

from chromadb import QueryResult
from langchain_core.messages.human import HumanMessage
from langchain_core.messages.system import SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.constants import START, END
from langgraph.graph.message import MessagesState
import chromadb
from langgraph.graph.state import StateGraph
from pydantic import Field
from rich import print as rprint
from init import model, system_prompt

chromadb = chromadb.PersistentClient("../chroma_db")
collection = chromadb.get_collection(name="FAQ_KNOWLEDGE_BASE")


class OverAllState(MessagesState):
    input_str: Annotated[str, add]
    retrieval_res: Annotated[QueryResult, "检索结果"]
    llm_output: Annotated[str, Field(description="模型输出")]

def retrieval_node(state: OverAllState) -> OverAllState:
    input_str = state["input_str"]

    retrieval_res = collection.query(
    query_texts=[input_str], # 查询文本
    n_results=3                     # 返回最相似的两个结果
    )
    rprint(retrieval_res)
    return {
        "retrieval_res": retrieval_res
    }

def llm_node(state: OverAllState) -> OverAllState:
    retrieval_res = state["retrieval_res"]
    input_str = state["input_str"]

    context = "\n\n".join(
        f"[文档 {i + 1}] {item['text']}" for i, item in enumerate(retrieval_res)
    )

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(
            content=f"请严格依据下面检索到的资料回答用户问题，资料中没有的内容不要编造。\n\n"
                    f"【检索资料】\n{context}\n\n"
                    f"【用户问题】\n{input_str}"
        )
    ]

    response = model.invoke(
        messages
    )
    llm_output = response["messages"][-1].content

    return {
        "llm_output": llm_output
    }

builder = StateGraph(state_schema=OverAllState)

builder.add_node("retrieval_node", retrieval_node)
builder.add_node("llm_node", llm_node)
builder.add_edge(START, "retrieval_node")
builder.add_edge("retrieval_node", "llm_node")
builder.add_edge("llm_node", END)

checkpointer = InMemorySaver()
config = {"configurable": {"thread_id": "123"}}
graph = builder.compile(checkpointer = checkpointer)

response = graph.invoke({"input_str": "明天吃什么"},config = config)

rprint(response)
